# Tema 1 · Clasificación — Score de Riesgo Crediticio
## Ejercicio del sábado (plantilla) · Dataset: **Credit Score Classification**

**Caso de negocio.** Una entidad financiera quiere clasificar a sus clientes en
tres niveles de score crediticio — **Poor**, **Standard**, **Good** — a partir
de su información financiera y de comportamiento de pago. Tu tarea es construir,
paso a paso, un modelo de **clasificación multiclase** que prediga el
`Credit_Score` de un cliente.

> **Cómo se evalúa.** El notebook mismo es la entrega. Completa las celdas
> marcadas con `# TODO:` siguiendo la instrucción en Markdown de cada sección.
> Las **secciones 1 y 2 ya están resueltas** como ejemplo del formato esperado.
> Al final respondes 2–3 preguntas de interpretación de negocio.

Estructura (misma que la demo del viernes):
1. Carga de datos y exploración inicial ✅ *(resuelto)*
2. Limpieza y EDA ✅ *(resuelto — este dataset viene "sucio")*
3. Preprocesamiento — **TODO**
4. Train/test split — **TODO**
5. Entrenamiento de modelos — **TODO**
6. Evaluación — **TODO**
7. Interpretación — **TODO**
8. Conclusión de negocio — **preguntas abiertas**


## 0. Semilla personal (anti-copia)

Reemplaza el valor por **tu número de cédula completo**. Esta semilla se usa en
el muestreo del dataset y en el `random_state` de tus splits y modelos, de modo
que tus resultados sean únicos.

In [ ]:
cedula = 1020304050  # <-- REEMPLAZA por tu número de cédula completo
import numpy as np
np.random.seed(cedula)
print(f"Semilla fijada en {cedula}")

## 1. Carga de datos y exploración inicial  ✅ *(resuelto)*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("../../data/01-clasificacion/credit_score_classification.csv",
                 low_memory=False)
print("Dimensiones originales:", df.shape)
df.head()

Tomamos una **muestra del 85%** con tu semilla. En un dataset de 100k filas
esto también agiliza el entrenamiento.

In [ ]:
df = df.sample(frac=0.85, random_state=cedula).reset_index(drop=True)
print("Dimensiones tras el muestreo:", df.shape)
df.info()

## 2. Limpieza y EDA  ✅ *(resuelto — estúdialo con atención)*

Este dataset es **realista y sucio**: números almacenados como texto con basura
(`28_`), valores imposibles (edades negativas), y placeholders como `_______`,
`_` o `!@9#%8` en las categóricas. Aquí dejamos resuelta una **limpieza base**;
tú la usarás en las siguientes secciones.

### 2.1 Columnas que no aportan al modelo
`ID`, `Customer_ID`, `Name`, `SSN` son identificadores → se descartan.

In [ ]:
drop_cols = ["ID", "Customer_ID", "Name", "SSN", "Month"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print("Columnas restantes:", list(df.columns))

### 2.2 Limpieza de numéricas guardadas como texto
Quitamos guiones bajos y convertimos a número. Los valores no convertibles pasan
a `NaN`; los negativos imposibles (edad, ingresos) también.

In [ ]:
num_like = ["Age", "Annual_Income", "Num_of_Loan", "Num_of_Delayed_Payment",
            "Outstanding_Debt", "Amount_invested_monthly", "Monthly_Balance",
            "Changed_Credit_Limit", "Num_Credit_Inquiries"]

for c in num_like:
    if c in df.columns:
        df[c] = (df[c].astype(str)
                       .str.replace("_", "", regex=False)
                       .str.strip())
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Edad válida entre 18 y 100; ingresos no negativos
df.loc[(df["Age"] < 18) | (df["Age"] > 100), "Age"] = np.nan
df.loc[df["Annual_Income"] < 0, "Annual_Income"] = np.nan
print("Numéricas limpiadas. Ejemplo Age describe:")
print(df["Age"].describe())

### 2.3 Placeholders basura en categóricas → NaN

In [ ]:
cat_clean = {"Occupation": "_______", "Credit_Mix": "_",
             "Payment_Behaviour": "!@9#%8", "Payment_of_Min_Amount": "NM"}
for col, junk in cat_clean.items():
    if col in df.columns:
        df[col] = df[col].replace(junk, np.nan)
print("Nulos por columna tras la limpieza:")
print(df.isna().sum()[df.isna().sum() > 0])

### 2.4 Distribución del target
El target `Credit_Score` tiene **3 clases** y está desbalanceado.

In [ ]:
ax = df["Credit_Score"].value_counts().plot(kind="bar",
        color=["#264653", "#2a9d8f", "#e9c46a"])
ax.set_title("Distribución del target (Credit_Score)")
ax.set_ylabel("Nº de clientes"); plt.xticks(rotation=0)
plt.show()
print(df["Credit_Score"].value_counts(normalize=True).round(3))

---
# A partir de aquí, completa tú el análisis (secciones 3–8)
---

## 3. Preprocesamiento  — **TODO**

Prepara los datos para el modelo. Sugerencias:
- Separa `X` (features) e `y` (target `Credit_Score`).
- Identifica columnas **numéricas** y **categóricas** restantes.
- Imputa los nulos (p. ej. mediana para numéricas, moda o categoría `"none"`
  para categóricas). Puedes usar `SimpleImputer` dentro de un `ColumnTransformer`.
- Codifica las categóricas (**OneHotEncoder**) y escala las numéricas
  (**StandardScaler**).

> Pista: reutiliza la idea del `ColumnTransformer` de la demo del viernes,
> añadiéndole imputación.

In [ ]:
# TODO: separa X e y
# X = ...
# y = ...

# TODO: define listas de columnas numéricas y categóricas

# TODO: construye un ColumnTransformer con imputación + OneHot (cat)
#       e imputación + StandardScaler (num)


## 4. Train/test split  — **TODO**

Divide en entrenamiento y prueba. Usa `stratify=y` (clases desbalanceadas) y
`random_state=cedula`.

In [ ]:
# TODO: from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(...)


## 5. Entrenamiento de modelos  — **TODO**

Entrena al menos **dos** modelos dentro de un `Pipeline` (preprocesamiento +
clasificador), por ejemplo:
- `LogisticRegression(max_iter=1000, class_weight="balanced")` como baseline.
- `RandomForestClassifier(class_weight="balanced")` como modelo principal.

Recuerda pasar `random_state=cedula`.

In [ ]:
# TODO: crea los pipelines y entrénalos con X_train, y_train


## 6. Evaluación  — **TODO**

Evalúa ambos modelos sobre el test. Para multiclase desbalanceada reporta:
- `classification_report` (precision / recall / f1 por clase).
- **F1 macro** (`f1_score(..., average="macro")`).
- Matriz de confusión (`confusion_matrix` + heatmap).

¿Cuál modelo se comporta mejor y por qué?

In [ ]:
# TODO: predice, imprime classification_report, F1 macro y matriz de confusión


## 7. Interpretación  — **TODO**

Extrae la **importancia de variables** del Random Forest (recuerda recuperar los
nombres tras el OneHot con `get_feature_names_out`). Grafica el top 15.

¿Qué variables pesan más en la predicción del score?

In [ ]:
# TODO: obtén y grafica la importancia de variables


## 8. Conclusión de negocio  — **responde en texto**

Responde brevemente (2–4 frases cada una):

1. **¿Qué variables tienen mayor peso en la predicción del score crediticio, y
   tiene sentido desde el negocio?**

   _(tu respuesta aquí)_

2. **Si el banco quisiera reducir los falsos negativos — clasificar como *Good* a
   alguien que en realidad es *Poor* — ¿qué ajustarías del modelo (umbral, pesos
   de clase, métricas objetivo)?**

   _(tu respuesta aquí)_

3. **¿Qué parte de la limpieza de datos crees que más impactó el resultado del
   modelo, y por qué?**

   _(tu respuesta aquí)_
